# Exercise 3.1: Ingestion Patterns and Concurrency

In this exercise, you'll learn about different write patterns in Apache Iceberg:
- **Batch vs Streaming**: Understand different ingestion patterns
- **Copy-on-Write vs Merge-on-Read**: Compare performance tradeoffs
- **Row-Level Operations**: Updates, deletes, and merges
- **Concurrency**: Handle concurrent writes and conflicts

## Learning Objectives
- Implement batch and streaming-style writes
- Compare COW and MOR strategies
- Use MERGE operations for upserts
- Understand and resolve write conflicts
- See how Iceberg handles concurrent modifications

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("IngestionPatterns") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.ingestion")
print("Namespace 'ingestion' created!")

## Part 1: Batch vs Streaming Ingestion Patterns

### Create Tables for Testing

In [ ]:
# Create a sensor readings table
spark.sql("""
CREATE OR REPLACE TABLE polaris.ingestion.sensor_readings (
    sensor_id STRING,
    reading_timestamp TIMESTAMP,
    temperature DOUBLE,
    humidity DOUBLE,
    reading_date DATE
)
USING iceberg
PARTITIONED BY (reading_date)
""")

print("Sensor readings table created!")

### Batch Ingestion Pattern

Large infrequent writes - typical for daily ETL jobs.

In [ ]:
# Simulate a batch load - one large insert
from datetime import datetime, timedelta
import random

# Generate batch data (1000 records)
batch_data = []
base_date = datetime(2024, 1, 1)
sensors = [f"sensor_{i}" for i in range(1, 11)]  # 10 sensors

for i in range(1000):
    ts = base_date + timedelta(hours=i % 24, minutes=random.randint(0, 59))
    batch_data.append((
        random.choice(sensors),
        ts,
        round(random.uniform(15.0, 35.0), 2),  # temperature
        round(random.uniform(30.0, 70.0), 2),  # humidity
        ts.date()
    ))

# Create DataFrame
batch_df = spark.createDataFrame(batch_data, 
    ["sensor_id", "reading_timestamp", "temperature", "humidity", "reading_date"])

print(f"Generated {batch_df.count()} records for batch insert")

In [ ]:
# Batch write - single large commit
start = time.time()

batch_df.writeTo("polaris.ingestion.sensor_readings") \
    .append()

batch_time = time.time() - start
print(f"Batch write completed in {batch_time:.2f} seconds")

In [ ]:
# Check snapshots - should be 1 new snapshot
print("Snapshots after batch write:")
spark.sql("""
SELECT 
    committed_at,
    operation,
    summary['added-records'] as records_added
FROM polaris.ingestion.sensor_readings.snapshots
ORDER BY committed_at
""").show()

### Streaming Ingestion Pattern

Small frequent writes - typical for real-time streaming.

In [ ]:
# Simulate streaming writes - many small commits
streaming_batches = 10
records_per_batch = 50

print(f"Simulating {streaming_batches} streaming micro-batches...")
streaming_times = []

base_date = datetime(2024, 1, 2)  # Next day

for batch_num in range(streaming_batches):
    # Generate small batch
    micro_batch = []
    for i in range(records_per_batch):
        ts = base_date + timedelta(minutes=batch_num * 10 + i)
        micro_batch.append((
            random.choice(sensors),
            ts,
            round(random.uniform(15.0, 35.0), 2),
            round(random.uniform(30.0, 70.0), 2),
            ts.date()
        ))
    
    micro_df = spark.createDataFrame(micro_batch,
        ["sensor_id", "reading_timestamp", "temperature", "humidity", "reading_date"])
    
    # Write micro-batch
    start = time.time()
    micro_df.writeTo("polaris.ingestion.sensor_readings").append()
    elapsed = time.time() - start
    streaming_times.append(elapsed)
    
    if (batch_num + 1) % 5 == 0:
        print(f"  Completed {batch_num + 1}/{streaming_batches} micro-batches")

print(f"\nStreaming writes completed!")
print(f"Average time per micro-batch: {sum(streaming_times)/len(streaming_times):.2f} seconds")

In [ ]:
# Check snapshots - should be many more now
print("Snapshots after streaming writes:")
snapshot_count = spark.sql("""
SELECT COUNT(*) as count
FROM polaris.ingestion.sensor_readings.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

# Show recent snapshots
spark.sql("""
SELECT 
    committed_at,
    operation,
    summary['added-records'] as records_added
FROM polaris.ingestion.sensor_readings.snapshots
ORDER BY committed_at DESC
LIMIT 5
""").show()

**Observation**: Streaming creates many small snapshots vs batch's single large snapshot.

## Part 2: Copy-on-Write vs Merge-on-Read

### Create Test Tables

In [ ]:
# Create two identical tables - one for COW, one for MOR
modes = {
    'cow': 'copy-on-write',
    'mor': 'merge-on-read'
}

for table_suffix, mode_value in modes.items():
    spark.sql(f"""
    CREATE OR REPLACE TABLE polaris.ingestion.products_{table_suffix} (
        product_id INT,
        product_name STRING,
        price DOUBLE,
        stock_quantity INT,
        last_updated TIMESTAMP
    )
    USING iceberg
    TBLPROPERTIES (
        'write.update.mode' = '{mode_value}',
        'write.delete.mode' = '{mode_value}',
        'write.merge.mode' = '{mode_value}'
    )
    """)

print("COW and MOR tables created!")

In [ ]:
# Insert initial data into both tables
initial_data = """
INSERT INTO polaris.ingestion.products_{mode} VALUES
    (1, 'Laptop', 1200.00, 50, TIMESTAMP '2024-01-01 00:00:00'),
    (2, 'Mouse', 25.00, 200, TIMESTAMP '2024-01-01 00:00:00'),
    (3, 'Keyboard', 75.00, 150, TIMESTAMP '2024-01-01 00:00:00'),
    (4, 'Monitor', 350.00, 80, TIMESTAMP '2024-01-01 00:00:00'),
    (5, 'Headphones', 89.99, 120, TIMESTAMP '2024-01-01 00:00:00')
"""

for mode in ['cow', 'mor']:
    spark.sql(initial_data.format(mode=mode))

print("Initial data inserted into both tables!")

### Compare UPDATE Performance

In [ ]:
# Update a few rows in COW mode
start = time.time()

spark.sql("""
UPDATE polaris.ingestion.products_cow
SET price = price * 1.10, last_updated = CURRENT_TIMESTAMP()
WHERE product_id IN (1, 3, 5)
""")

cow_time = time.time() - start
print(f"COW UPDATE took {cow_time:.3f} seconds")

In [ ]:
# Same update in MOR mode
start = time.time()

spark.sql("""
UPDATE polaris.ingestion.products_mor
SET price = price * 1.10, last_updated = CURRENT_TIMESTAMP()
WHERE product_id IN (1, 3, 5)
""")

mor_time = time.time() - start
print(f"MOR UPDATE took {mor_time:.3f} seconds")
print(f"\nMOR was {cow_time/mor_time:.1f}x faster")

### Examine File Structure

In [ ]:
# COW creates new data files
print("COW Files:")
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes,
    record_count
FROM polaris.ingestion.products_cow.files
""").show(truncate=False)

In [ ]:
# MOR creates delete files
print("MOR Files:")
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes,
    record_count
FROM polaris.ingestion.products_mor.files
""").show(truncate=False)

In [ ]:
# Check for delete files in MOR
print("MOR Delete Files:")
spark.sql("""
SELECT 
    file_path,
    file_size_in_bytes,
    record_count
FROM polaris.ingestion.products_mor.delete_files
""").show(truncate=False)

**Key Difference**: 
- COW rewrote entire data files
- MOR created small delete files pointing to deleted rows

### Compare READ Performance

In [ ]:
# Read from COW table
start = time.time()
cow_result = spark.sql("SELECT * FROM polaris.ingestion.products_cow").collect()
cow_read_time = time.time() - start

print(f"COW READ took {cow_read_time:.3f} seconds")

In [ ]:
# Read from MOR table
start = time.time()
mor_result = spark.sql("SELECT * FROM polaris.ingestion.products_mor").collect()
mor_read_time = time.time() - start

print(f"MOR READ took {mor_read_time:.3f} seconds")
print(f"\nCOW reads were {mor_read_time/cow_read_time:.1f}x faster (MOR has merge overhead)")

## Part 3: Row-Level Operations

### DELETE Operations

In [ ]:
# Metadata-only delete (partition-level)
spark.sql("""
CREATE OR REPLACE TABLE polaris.ingestion.orders (
    order_id INT,
    customer_id STRING,
    order_date DATE,
    amount DOUBLE
)
USING iceberg
PARTITIONED BY (order_date)
""")

# Insert data across multiple partitions
spark.sql("""
INSERT INTO polaris.ingestion.orders VALUES
    (1, 'CUST001', DATE '2024-01-01', 100.00),
    (2, 'CUST002', DATE '2024-01-01', 150.00),
    (3, 'CUST003', DATE '2024-01-02', 200.00),
    (4, 'CUST004', DATE '2024-01-02', 250.00),
    (5, 'CUST005', DATE '2024-01-03', 300.00)
""")

print("Orders table created and populated!")

In [ ]:
# Partition-level delete (very fast - metadata only)
start = time.time()

spark.sql("""
DELETE FROM polaris.ingestion.orders
WHERE order_date = DATE '2024-01-01'
""")

delete_time = time.time() - start
print(f"Partition delete took {delete_time:.3f} seconds (metadata-only!)")

In [ ]:
# Verify deletion
spark.sql("SELECT * FROM polaris.ingestion.orders ORDER BY order_id").show()

### MERGE Operations (Upserts)

In [ ]:
# Create a products table
spark.sql("""
CREATE OR REPLACE TABLE polaris.ingestion.inventory (
    product_id INT,
    product_name STRING,
    quantity INT,
    last_updated TIMESTAMP
)
USING iceberg
""")

# Initial inventory
spark.sql("""
INSERT INTO polaris.ingestion.inventory VALUES
    (1, 'Widget A', 100, TIMESTAMP '2024-01-01 00:00:00'),
    (2, 'Widget B', 200, TIMESTAMP '2024-01-01 00:00:00'),
    (3, 'Widget C', 150, TIMESTAMP '2024-01-01 00:00:00')
""")

print("Inventory table created!")

In [ ]:
# Create updates (some existing, some new)
updates_df = spark.createDataFrame([
    (1, 'Widget A', 120, datetime(2024, 1, 2)),  # Update existing
    (2, 'Widget B', 180, datetime(2024, 1, 2)),  # Update existing  
    (4, 'Widget D', 75, datetime(2024, 1, 2)),   # Insert new
    (5, 'Widget E', 90, datetime(2024, 1, 2))    # Insert new
], ["product_id", "product_name", "quantity", "last_updated"])

updates_df.createOrReplaceTempView("updates")

print("Updates prepared!")

In [ ]:
# Perform MERGE (upsert)
spark.sql("""
MERGE INTO polaris.ingestion.inventory t
USING updates s
ON t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET
    t.quantity = s.quantity,
    t.last_updated = s.last_updated
WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE completed!")

In [ ]:
# View results
print("Inventory after MERGE:")
spark.sql("""
SELECT * FROM polaris.ingestion.inventory 
ORDER BY product_id
""").show()

## Part 4: Concurrency and Conflicts

### Create Test Table

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE polaris.ingestion.accounts (
    account_id INT,
    account_name STRING,
    balance DOUBLE,
    region STRING
)
USING iceberg
PARTITIONED BY (region)
""")

spark.sql("""
INSERT INTO polaris.ingestion.accounts VALUES
    (1, 'Account A', 1000.00, 'EAST'),
    (2, 'Account B', 2000.00, 'EAST'),
    (3, 'Account C', 1500.00, 'WEST'),
    (4, 'Account D', 3000.00, 'WEST')
""")

print("Accounts table created!")

### Non-Conflicting Writes

Updates to different partitions don't conflict.

In [ ]:
# This would work if run concurrently (different partitions)
print("Updating EAST region...")
spark.sql("""
UPDATE polaris.ingestion.accounts
SET balance = balance * 1.05
WHERE region = 'EAST'
""")

print("Updating WEST region...")
spark.sql("""
UPDATE polaris.ingestion.accounts
SET balance = balance * 1.05
WHERE region = 'WEST'
""")

print("Both updates succeeded (different partitions)!")

### Conflicting Writes

Updates to the same partition will conflict.

In [ ]:
# Simulate conflict by trying to update same partition
# In real concurrent scenario, one would fail

print("This demonstrates what WOULD happen with concurrent conflicting writes:")
print("")
print("Writer 1: UPDATE accounts SET balance = balance + 100 WHERE region = 'EAST'")
print("Writer 2: UPDATE accounts SET balance = balance * 1.1 WHERE region = 'EAST'")
print("")
print("Result: One writer would get a CommitFailedException")
print("The failed writer would need to retry with fresh data")

### Understanding ON Clause in MERGE

In [ ]:
# The ON clause determines what Iceberg checks for conflicts
print("MERGE isolation depends on ON clause:")
print("")
print("Good - Includes partition column:")
print("MERGE INTO accounts t USING updates s")
print("ON t.account_id = s.account_id AND t.region = s.region")
print("→ Iceberg can isolate by partition")
print("")
print("Careful - No partition column:")
print("MERGE INTO accounts t USING updates s") 
print("ON t.account_id = s.account_id")
print("→ Iceberg must check all partitions, more conflicts possible")

## Key Takeaways

### Ingestion Patterns
1. **Batch**: Large infrequent writes, fewer snapshots
2. **Streaming**: Small frequent writes, many snapshots (needs more maintenance)
3. **Both work**: Same table structure, different commit patterns

### COW vs MOR
1. **COW**: Slower writes, faster reads, rewrites entire files
2. **MOR**: Faster writes, slower reads, creates delete files
3. **Choose based on workload**: Update-heavy → MOR, Read-heavy → COW

### Row-Level Operations
1. **DELETE**: Partition-level deletes are metadata-only (fast!)
2. **UPDATE**: Performance depends on COW vs MOR
3. **MERGE**: Perfect for upserts and CDC patterns

### Concurrency
1. **Partition isolation**: Different partitions don't conflict
2. **Same partition**: Conflicts possible, automatic retry
3. **ON clause matters**: Include partition columns when possible

## Real-World Recommendations

- **High-volume streaming**: Use MOR, compact regularly
- **Analytical workloads**: Use COW for read performance  
- **Mixed workloads**: Use MOR + periodic compaction
- **Partition wisely**: Enable concurrent writers
- **Monitor snapshots**: Streaming creates many, expire regularly

## Challenge Exercise

1. Create a table with your own schema
2. Implement both batch and streaming patterns
3. Compare COW and MOR for your use case
4. Try MERGE operations with different ON clauses
5. Design a partitioning strategy that minimizes conflicts

## Cleanup

In [ ]:
# Optional: Drop tables
# for table in ['sensor_readings', 'products_cow', 'products_mor', 'orders', 'inventory', 'accounts']:
#     spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.{table}")
# print("Tables dropped!")